<a href="https://colab.research.google.com/github/OdysseusPolymetis/enexdi_prep_2026/blob/main/2_NLP_bis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Linguistic statistics from lemmas and POS tags**
---

This notebook introduces simple **corpus statistics** based on linguistic analysis with **Stanza**.

This version assumes that we work with **one single `.txt` file**.

The goal is to analyze one text automatically and produce interpretable statistics and visualizations:

1. lemma frequencies;
2. distribution of grammatical categories;
3. most frequent nouns, verbs, and adjectives;
4. concordances around a target lemma;
5. co-occurrences around a target lemma;
6. evolution of grammatical categories across the text;
7. lexical density across the text;
8. dispersion of selected lemmas;
9. sentence-length statistics;
10. export of the results.

The notebook uses:

- **Stanza** for tokenization, lemmatization, and POS tagging;
- a list of **French stopwords** downloaded from GitHub;
- `pandas` for tables;
- `matplotlib` for visualizations.


## 1. Installing the required libraries

In [ ]:
!pip -q install stanza pandas matplotlib tqdm

## 2. Imports

In [ ]:
from google.colab import files
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import stanza
from tqdm.auto import tqdm
import re
import zipfile

## 3. Upload one `.txt` file

Upload one text file from your computer.

For this exercise, we assume that only **one file** is used.


In [ ]:
uploaded = files.upload()

filename = list(uploaded.keys())[0]
path = Path("/content") / filename
path.write_bytes(uploaded[filename])

print("Uploaded file:", path.name)

## 4. Download the stopwords

Stopwords are very frequent words, often grammatical words, that we may want to remove from lexical analyses.

Here, we use a list of French stopwords provided for the course.


In [ ]:
!wget -q -O /content/stopwords_fr.txt https://github.com/OdysseusPolymetis/enexdi2025_prep/raw/refs/heads/main/stopwords_fr.txt

with open("/content/stopwords_fr.txt", "r", encoding="utf-8") as f:
    stopwords = set(line.strip().lower() for line in f if line.strip())

print("Number of stopwords:", len(stopwords))
print(list(stopwords)[:20])

## 5. Load Stanza

We use Stanza to:

- split the text into sentences;
- split sentences into words;
- obtain the lemma of each word;
- obtain its universal part-of-speech tag, or `UPOS`.

In this notebook, we work with French.


In [ ]:
language = "fr"

stanza.download(language)

nlp = stanza.Pipeline(
    lang=language,
    processors="tokenize,mwt,pos,lemma",
    use_gpu=True
)

## 6. Read and analyze the text with Stanza

We create a table in which each row corresponds to one analyzed word.

Each row contains:

- the sentence number;
- the word position in the text;
- the original word form;
- the lemma;
- the grammatical category.


In [ ]:
def read_text(path):
    return path.read_text(encoding="utf-8")

def split_into_chunks(text, chunk_size=20000):
    return [
        text[i:i+chunk_size]
        for i in range(0, len(text), chunk_size)
    ]

text = read_text(path)
chunks = split_into_chunks(text)

rows = []
global_position = 0
sentence_number = 0

for chunk in tqdm(chunks):
    doc = nlp(chunk)

    for sent in doc.sentences:
        sentence_number += 1

        for word in sent.words:
            rows.append({
                "sentence_id": sentence_number,
                "position": global_position,
                "form": word.text,
                "lemma": word.lemma.lower() if word.lemma else word.text.lower(),
                "upos": word.upos
            })

            global_position += 1

df = pd.DataFrame(rows)

print("Total number of analyzed tokens:", len(df))
df.head()

## 7. Minimal cleaning for lexical statistics

We create a filtered version of the table:

- without punctuation;
- without stopwords;
- without empty tokens;
- with only tokens containing at least one letter.

This version will be used for lexical statistics.


In [ ]:
def contains_a_letter(text):
    return bool(re.search(r"[A-Za-zÀ-ÖØ-öø-ÿ]", str(text)))

df_words = df[
    (df["upos"] != "PUNCT") &
    (df["lemma"].apply(contains_a_letter)) &
    (~df["lemma"].isin(stopwords))
].copy()

print("Number of tokens after filtering:", len(df_words))
df_words.head()

## 8. Lemma frequency

We count the most frequent lemmas in the text.

Lemmatization makes it possible to group several forms of the same word.


In [ ]:
lemma_freq = (
    df_words["lemma"]
    .value_counts()
    .reset_index()
)

lemma_freq.columns = ["lemma", "frequency"]

lemma_freq.head(30)

## 9. Visualization — most frequent lemmas

In [ ]:
top = lemma_freq.head(20)

plt.figure(figsize=(10, 6))
plt.barh(top["lemma"][::-1], top["frequency"][::-1])
plt.title("Most frequent lemmas")
plt.xlabel("Frequency")
plt.ylabel("Lemma")
plt.show()

## 10. Visualization — cumulative coverage of frequent lemmas

This plot shows how much of the filtered vocabulary is covered by the most frequent lemmas.

It helps us see whether the text is dominated by a small number of repeated lemmas or whether its vocabulary is more dispersed.


In [ ]:
top_n = 100

coverage = lemma_freq.head(top_n).copy()
coverage["cumulative_frequency"] = coverage["frequency"].cumsum()
coverage["coverage"] = coverage["cumulative_frequency"] / lemma_freq["frequency"].sum()

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(coverage) + 1), coverage["coverage"])
plt.title("Cumulative coverage of the most frequent lemmas")
plt.xlabel("Number of lemmas")
plt.ylabel("Share of filtered tokens")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.show()

## 11. Distribution of grammatical categories

We now observe the most frequent grammatical categories in the text.

This makes it possible to characterize the grammatical profile of the text: proportion of nouns, verbs, adjectives, adverbs, etc.


In [ ]:
pos_freq = (
    df[df["upos"] != "PUNCT"]["upos"]
    .value_counts()
    .reset_index()
)

pos_freq.columns = ["upos", "frequency"]
pos_freq["proportion"] = pos_freq["frequency"] / pos_freq["frequency"].sum()

pos_freq

## 12. Visualization — POS distribution

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(pos_freq["upos"], pos_freq["proportion"])
plt.title("Distribution of grammatical categories")
plt.xlabel("Grammatical category")
plt.ylabel("Proportion")
plt.xticks(rotation=45)
plt.show()

## 13. Visualization — POS distribution as a pie chart

A pie chart is less precise than a bar chart, but it can be useful for a quick overview of the grammatical composition of the text.


In [ ]:
top_pos = pos_freq.copy()
small_categories = top_pos[top_pos["proportion"] < 0.02]

if len(small_categories) > 0:
    main_categories = top_pos[top_pos["proportion"] >= 0.02].copy()
    other = pd.DataFrame({
        "upos": ["OTHER"],
        "frequency": [small_categories["frequency"].sum()],
        "proportion": [small_categories["proportion"].sum()]
    })
    pie_data = pd.concat([main_categories, other], ignore_index=True)
else:
    pie_data = top_pos

plt.figure(figsize=(8, 8))
plt.pie(
    pie_data["proportion"],
    labels=pie_data["upos"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Grammatical composition of the text")
plt.show()

## 14. The most frequent nouns, verbs, and adjectives

We can use POS tags to filter certain types of words.

This is often more interpretable than a simple list of frequent words.


In [ ]:
def top_lemmas_by_pos(pos, n=20):
    result = (
        df_words[df_words["upos"] == pos]["lemma"]
        .value_counts()
        .head(n)
        .reset_index()
    )

    result.columns = ["lemma", "frequency"]
    return result

top_nouns = top_lemmas_by_pos("NOUN")
top_verbs = top_lemmas_by_pos("VERB")
top_adjectives = top_lemmas_by_pos("ADJ")

print("Most frequent nouns")
display(top_nouns)

print("Most frequent verbs")
display(top_verbs)

print("Most frequent adjectives")
display(top_adjectives)

## 15. Visualization — nouns, verbs, and adjectives

These plots make it easier to compare the main lexical categories.


In [ ]:
def plot_top_table(table, title):
    plt.figure(figsize=(9, 5))
    plt.barh(table["lemma"][::-1], table["frequency"][::-1])
    plt.title(title)
    plt.xlabel("Frequency")
    plt.ylabel("Lemma")
    plt.show()

plot_top_table(top_nouns, "Most frequent nouns")
plot_top_table(top_verbs, "Most frequent verbs")
plot_top_table(top_adjectives, "Most frequent adjectives")

## 16. Combined lemma + POS frequencies

The same lemma may sometimes correspond to several grammatical categories.

We can therefore count `(lemma, POS)` pairs.


In [ ]:
lemma_pos_freq = (
    df_words
    .groupby(["lemma", "upos"])
    .size()
    .reset_index(name="frequency")
    .sort_values("frequency", ascending=False)
)

lemma_pos_freq.head(30)

## 17. Simple concordance around a lemma

A statistic should often be checked by going back to the text.

This cell displays the contexts of a target lemma.


In [ ]:
def concordance(df, target_lemma, window=6, max_results=20):
    target_lemma = target_lemma.lower()
    results = []

    df_sorted = df.sort_values("position").reset_index(drop=True)
    positions = df_sorted.index[df_sorted["lemma"] == target_lemma].tolist()

    for pos in positions[:max_results]:
        start = max(0, pos - window)
        end = min(len(df_sorted), pos + window + 1)

        left = " ".join(df_sorted.loc[start:pos-1, "form"])
        word = df_sorted.loc[pos, "form"]
        right = " ".join(df_sorted.loc[pos+1:end-1, "form"])

        results.append({
            "left_context": left,
            "word": word,
            "right_context": right
        })

    return pd.DataFrame(results)

concordance(df, "roi", window=7, max_results=20)

## 18. Co-occurrences around a target lemma

We look for words that often appear around a given lemma.

This makes it possible to observe the lexical environment of a word.


In [ ]:
def cooccurrences(df, target_lemma, window=5, selected_pos=None, n=30):
    target_lemma = target_lemma.lower()
    counter = Counter()

    df_sorted = df.sort_values("position").reset_index(drop=True)
    positions = df_sorted.index[df_sorted["lemma"] == target_lemma].tolist()

    for pos in positions:
        start = max(0, pos - window)
        end = min(len(df_sorted), pos + window + 1)

        context = df_sorted.loc[start:end-1]

        for _, row in context.iterrows():
            lemma = row["lemma"]
            upos = row["upos"]

            if lemma == target_lemma:
                continue

            if upos == "PUNCT":
                continue

            if lemma in stopwords:
                continue

            if selected_pos is not None and upos not in selected_pos:
                continue

            if contains_a_letter(lemma):
                counter[lemma] += 1

    return pd.DataFrame(
        counter.most_common(n),
        columns=["lemma", "frequency"]
    )

target_lemma = "roi"

cooc = cooccurrences(
    df,
    target_lemma,
    window=5,
    selected_pos=["NOUN", "VERB", "ADJ"],
    n=30
)

cooc

## 19. Visualization — co-occurrences around a target lemma

This graph shows the most frequent lexical neighbors of a target lemma.


In [ ]:
if len(cooc) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(cooc["lemma"][::-1], cooc["frequency"][::-1])
    plt.title(f"Co-occurrences around '{target_lemma}'")
    plt.xlabel("Frequency in context window")
    plt.ylabel("Lemma")
    plt.show()
else:
    print("No co-occurrence found for this lemma.")

## 20. Divide the text into segments

To observe changes inside a single text, we divide it into equal-sized segments.

This makes it possible to study internal variation: for example, whether nouns, verbs, or adjectives become more frequent in certain parts of the text.


In [ ]:
number_of_segments = 10

df["segment"] = pd.cut(
    df["position"],
    bins=number_of_segments,
    labels=[f"Segment {i+1}" for i in range(number_of_segments)]
)

df_words["segment"] = pd.cut(
    df_words["position"],
    bins=number_of_segments,
    labels=[f"Segment {i+1}" for i in range(number_of_segments)]
)

df[["position", "segment"]].head()

## 21. Visualization — POS evolution across the text

This stacked bar chart shows how grammatical categories vary across the text.


In [ ]:
main_pos = ["NOUN", "VERB", "ADJ", "ADV", "PROPN", "PRON", "DET", "ADP"]

segment_pos = (
    df[(df["upos"] != "PUNCT") & (df["upos"].isin(main_pos))]
    .groupby(["segment", "upos"], observed=False)
    .size()
    .reset_index(name="frequency")
)

segment_totals = (
    df[(df["upos"] != "PUNCT") & (df["upos"].isin(main_pos))]
    .groupby("segment", observed=False)
    .size()
    .reset_index(name="total")
)

segment_pos = segment_pos.merge(segment_totals, on="segment")
segment_pos["proportion"] = segment_pos["frequency"] / segment_pos["total"]

segment_pos_table = segment_pos.pivot_table(
    index="segment",
    columns="upos",
    values="proportion",
    fill_value=0,
    observed=False
)

segment_pos_table.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Evolution of POS categories across the text")
plt.xlabel("Text segment")
plt.ylabel("Proportion")
plt.legend(title="POS", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

## 22. Visualization — POS heatmap across the text

The same information can be displayed as a heatmap.

Darker cells correspond to higher proportions.


In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(segment_pos_table.T, aspect="auto")
plt.colorbar(label="Proportion")
plt.yticks(range(len(segment_pos_table.columns)), segment_pos_table.columns)
plt.xticks(range(len(segment_pos_table.index)), segment_pos_table.index, rotation=45)
plt.title("POS distribution across text segments")
plt.xlabel("Text segment")
plt.ylabel("POS tag")
plt.show()

## 23. Lexical density across the text

Lexical density is the proportion of content words.

Here, we use a simple definition: nouns, proper nouns, verbs, adjectives, and adverbs.


In [ ]:
content_pos = ["NOUN", "PROPN", "VERB", "ADJ", "ADV"]

density_by_segment = (
    df[df["upos"] != "PUNCT"]
    .assign(is_content_word=lambda x: x["upos"].isin(content_pos))
    .groupby("segment", observed=False)["is_content_word"]
    .mean()
    .reset_index(name="lexical_density")
)

density_by_segment

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(density_by_segment["segment"].astype(str), density_by_segment["lexical_density"], marker="o")
plt.title("Lexical density across the text")
plt.xlabel("Text segment")
plt.ylabel("Share of content words")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.show()

## 24. Relative frequency of a target lemma across the text

This visualization helps us see where a target lemma appears most often.


In [ ]:
target_lemma = "roi"

target_by_segment = (
    df_words
    .assign(is_target=lambda x: x["lemma"] == target_lemma)
    .groupby("segment", observed=False)["is_target"]
    .mean()
    .reset_index(name="relative_frequency")
)

target_by_segment["relative_frequency_per_1000"] = target_by_segment["relative_frequency"] * 1000

plt.figure(figsize=(10, 5))
plt.plot(
    target_by_segment["segment"].astype(str),
    target_by_segment["relative_frequency_per_1000"],
    marker="o"
)
plt.title(f"Relative frequency of '{target_lemma}' across the text")
plt.xlabel("Text segment")
plt.ylabel("Frequency per 1,000 filtered tokens")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.show()

## 25. Dispersion of selected lemmas

A dispersion plot shows where selected words appear in the text.

Each point corresponds to one occurrence.


In [ ]:
lemmas_to_plot = ["valjean", "madeleine", "fauchelevent", "javert"]

plt.figure(figsize=(12, 4))

for i, lemma in enumerate(lemmas_to_plot):
    positions = df_words[df_words["lemma"] == lemma]["position"]
    plt.scatter(positions, [i] * len(positions), label=lemma, s=20)

plt.yticks(range(len(lemmas_to_plot)), lemmas_to_plot)
plt.xlabel("Position in the text")
plt.ylabel("Lemma")
plt.title("Dispersion of selected lemmas")
plt.grid(True, axis="x", alpha=0.3)
plt.show()

## 26. Heatmap of frequent lemmas across text segments

This heatmap shows where the most frequent lemmas are concentrated in the text.

Each cell gives the relative frequency of a lemma in a segment.


In [ ]:
top_lemmas_for_heatmap = lemma_freq.head(15)["lemma"].tolist()

lemma_segment = (
    df_words[df_words["lemma"].isin(top_lemmas_for_heatmap)]
    .groupby(["lemma", "segment"], observed=False)
    .size()
    .reset_index(name="frequency")
)

segment_sizes = (
    df_words
    .groupby("segment", observed=False)
    .size()
    .reset_index(name="segment_size")
)

lemma_segment = lemma_segment.merge(segment_sizes, on="segment")
lemma_segment["freq_per_1000"] = lemma_segment["frequency"] / lemma_segment["segment_size"] * 1000

lemma_segment_table = lemma_segment.pivot_table(
    index="lemma",
    columns="segment",
    values="freq_per_1000",
    fill_value=0,
    observed=False
)

plt.figure(figsize=(12, 7))
plt.imshow(lemma_segment_table, aspect="auto")
plt.colorbar(label="Frequency per 1,000 filtered tokens")
plt.yticks(range(len(lemma_segment_table.index)), lemma_segment_table.index)
plt.xticks(range(len(lemma_segment_table.columns)), lemma_segment_table.columns, rotation=45)
plt.title("Frequent lemmas across text segments")
plt.xlabel("Text segment")
plt.ylabel("Lemma")
plt.show()

## 27. Sentence length

We can also produce simple statistics on sentence length.

Sentence length can provide clues about style, genre, or the structure of a text.


In [ ]:
sentence_lengths = (
    df[df["upos"] != "PUNCT"]
    .groupby("sentence_id")
    .size()
    .reset_index(name="length")
)

sentence_lengths["length"].describe()

## 28. Visualization — sentence length distribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(sentence_lengths["length"], bins=30)
plt.title("Distribution of sentence lengths")
plt.xlabel("Number of words")
plt.ylabel("Number of sentences")
plt.show()

## 29. Visualization — sentence length boxplot

In [ ]:
plt.figure(figsize=(6, 5))
plt.boxplot(sentence_lengths["length"], vert=True)
plt.title("Sentence length boxplot")
plt.ylabel("Number of words")
plt.xticks([1], ["Text"])
plt.show()